In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

class CFG:
    ROOT       = os.path.join(os.path.expanduser("~"), "pig_posture_recognition")
    TRAIN_IMG  = os.path.join(ROOT, "train_images")
    TEST_IMG   = os.path.join(ROOT, "test_images")
    TRAIN_CSV  = os.path.join(ROOT, "train.csv")
    TEST_CSV   = os.path.join(ROOT, "test.csv")
    SAMPLE_SUB = os.path.join(ROOT, "sample_submission.csv")
    OUTPUT     = os.path.join(os.path.expanduser("~"), "working")
    os.makedirs(OUTPUT, exist_ok=True)

    # ── Experiment ───────────────────────────────────────────────────────────
    SEED       = 42
    N_FOLDS    = 5
    TRAIN_FOLDS= [0, 1, 2, 3, 4]
    DEBUG      = False

    # ── Crop ─────────────────────────────────────────────────────────────────
    IMG_SIZE   = 320            # ↑ from 224 → better posture detail
    PAD_RATIO  = 0.20           # ↑ from 0.15 → more context

    # ── Models ───────────────────────────────────────────────────────────────
    MODELS = [
        {"name": "tf_efficientnetv2_m",           "img_size": 320, "weight": 0.40},
        {"name": "convnext_base.in12k",            "img_size": 320, "weight": 0.35},
        {"name": "swin_small_patch4_window7_224",  "img_size": 224, "weight": 0.25},
    ]
    NUM_CLASSES  = 5
    DROP_RATE    = 0.20         # ↓ from 0.30
    DROP_PATH    = 0.20

    # ── Training ─────────────────────────────────────────────────────────────
    EPOCHS       = 30           # ↑ from 25
    BATCH_SIZE   = 32           # ↓ for 320px
    ACCUM_STEPS  = 2            # effective batch = 64
    LR           = 2e-4
    MIN_LR       = 1e-6
    WEIGHT_DECAY = 1e-2
    WARMUP_EPOCHS= 3
    LABEL_SMOOTH = 0.03         # ↓ from 0.05
    MIXUP_ALPHA  = 0.40         # MixUp regularization

    # ── Model Soup ───────────────────────────────────────────────────────────
    SOUP_EPOCHS  = 5            # average last N epoch checkpoints

    # ── TTA ──────────────────────────────────────────────────────────────────
    TTA_STEPS    = 8            # ↑ from 5

    # ── Pseudo labeling ──────────────────────────────────────────────────────
    PSEUDO_ROUNDS      = [0.98, 0.95]  # confidence thresholds per round

    # ── DataLoader ───────────────────────────────────────────────────────────
    NUM_WORKERS  = 4
    PIN_MEMORY   = True
    MIXED_PREC   = True

    CLASS_NAMES  = [
        "Lateral_lying_left",   # 0
        "Lateral_lying_right",  # 1
        "Sitting",              # 2
        "Standing",             # 3
        "Sternal_lying",        # 4
    ]

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(CFG.SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

Device : cuda


In [ ]:
def parse_bbox(bbox_str):
    cleaned = str(bbox_str).strip().replace("[","").replace("]","")
    parts   = [float(x) for x in cleaned.split(",")]
    return parts[0], parts[1], parts[2], parts[3]

def load_dataframes():
    train_df = pd.read_csv(CFG.TRAIN_CSV)
    test_df  = pd.read_csv(CFG.TEST_CSV)
    for df in [train_df, test_df]:
        coords = df["bbox"].apply(parse_bbox)
        df[["xmin","ymin","w","h"]] = pd.DataFrame(coords.tolist(), index=df.index)
    print(f"Train : {len(train_df):,} instances")
    print(f"Test  : {len(test_df):,} instances")
    print("\nClass distribution:")
    print(train_df["class_id"].value_counts().sort_index()
          .rename(index={i: f"{i}-{n}" for i,n in enumerate(CFG.CLASS_NAMES)}))
    if CFG.DEBUG:
        train_df = train_df.sample(300, random_state=CFG.SEED).reset_index(drop=True)
    return train_df, test_df

train_df, test_df = load_dataframes()

Train : 16,062 instances
Test  : 6,872 instances

Class distribution:
class_id
0-Lateral_lying_left     2114
1-Lateral_lying_right    2360
2-Sitting                 469
3-Standing               6729
4-Sternal_lying          4390
Name: count, dtype: int64


In [ ]:
def add_fold_column(df: pd.DataFrame) -> pd.DataFrame:
    # Split by image_id to avoid leakage
    img_cls = (df.groupby("image_id")["class_id"]
                 .agg(lambda x: x.mode()[0])
                 .reset_index()
                 .rename(columns={"class_id": "img_class"}))
    skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    img_cls["fold"] = -1
    for fold, (_, vi) in enumerate(skf.split(img_cls, img_cls["img_class"])):
        img_cls.loc[vi, "fold"] = fold
    if "fold" in df.columns:
        df = df.drop(columns=["fold"])
    df = df.merge(img_cls[["image_id","fold"]], on="image_id", how="left")
    print("Fold distribution:")
    print(df["fold"].value_counts().sort_index())
    return df

train_df = add_fold_column(train_df)

Fold distribution:
fold
0    3216
1    3231
2    3192
3    3230
4    3193
Name: count, dtype: int64


In [ ]:
def get_train_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size),
                            scale=(0.70, 1.0), ratio=(0.75, 1.33), p=1.0),
        # ✅ NO HorizontalFlip — it swaps Lateral_left ↔ Lateral_right labels!
        # ✅ NO VerticalFlip   — unrealistic for farm overhead cameras
        A.Rotate(limit=20, p=0.50),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.60),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30,
                             val_shift_limit=20, p=0.50),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8,8), p=0.30),
        A.OneOf([
            A.MotionBlur(blur_limit=7),
            A.GaussianBlur(blur_limit=(3,7)),
            A.MedianBlur(blur_limit=5),
        ], p=0.30),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.30),
        A.CoarseDropout(max_holes=8,
                        max_height=img_size//8, max_width=img_size//8,
                        min_holes=1, fill_value=0, p=0.30),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size: int) -> A.Compose:
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.85, 1.0), p=1.0),
        A.Rotate(limit=10, p=0.50),
        A.RandomBrightnessContrast(brightness_limit=0.10,
                                   contrast_limit=0.10, p=0.50),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

print("Augmentation pipelines defined ✓")

Augmentation pipelines defined ✓


In [ ]:
def crop_with_padding(img, xmin, ymin, w, h, pad_ratio=CFG.PAD_RATIO):
    H, W  = img.shape[:2]
    px, py = int(w * pad_ratio), int(h * pad_ratio)
    x1 = max(0, int(xmin) - px);  y1 = max(0, int(ymin) - py)
    x2 = min(W, int(xmin+w) + px); y2 = min(H, int(ymin+h) + py)
    crop = img[y1:y2, x1:x2]
    if crop.size == 0:
        crop = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
    return crop

class PigDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, row.image_id))
        img = np.zeros((256,256,3),dtype=np.uint8) if img is None \
              else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        crop = crop_with_padding(img, row.xmin, row.ymin, row.w, row.h)
        if crop.shape[0] < 16 or crop.shape[1] < 16:
            crop = cv2.resize(crop, (64,64))
        if self.transform:
            crop = self.transform(image=crop)["image"]
        if self.is_test:
            return crop, row.row_id
        return crop, int(row.class_id)

print("Dataset defined ✓")

Dataset defined ✓


In [ ]:
class PigClassifier(nn.Module):
    def __init__(self, model_name, num_classes,
                 drop_rate=CFG.DROP_RATE,
                 drop_path_rate=CFG.DROP_PATH,
                 pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool="avg",
            drop_rate=drop_rate, drop_path_rate=drop_path_rate,
        )
        in_feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_feat),
            nn.Dropout(drop_rate),
            nn.Linear(in_feat, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

print("Model defined ✓")

Model defined ✓


In [ ]:
# ── Label smoothing loss ──────────────────────────────────────────────────────
class LabelSmoothingCE(nn.Module):
    def __init__(self, classes, smoothing=0.03):
        super().__init__()
        self.smoothing = smoothing
        self.n = classes
    def forward(self, logits, targets):
        conf  = 1.0 - self.smoothing
        other = self.smoothing / (self.n - 1)
        onehot = torch.full_like(logits, other)
        onehot.scatter_(1, targets.unsqueeze(1), conf)
        return -(onehot * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

# ── MixUp ─────────────────────────────────────────────────────────────────────
def mixup_data(x, y, alpha=CFG.MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1-lam) * criterion(pred, y_b)

# ── Weighted sampler ──────────────────────────────────────────────────────────
def make_weighted_sampler(labels):
    counts  = Counter(labels)
    w_map   = {c: 1.0/n for c,n in counts.items()}
    weights = torch.tensor([w_map[l] for l in labels], dtype=torch.double)
    return WeightedRandomSampler(weights, len(weights), replacement=True)

# ── Warmup + Cosine LR ────────────────────────────────────────────────────────
class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=CFG.MIN_LR):
        self.opt    = optimizer
        self.warmup = warmup_epochs
        self.total  = total_epochs
        self.min_lr = min_lr
        self.base   = [g["lr"] for g in optimizer.param_groups]
        self._step  = 0
    def step(self):
        self._step += 1
        e = self._step
        for base_lr, grp in zip(self.base, self.opt.param_groups):
            if e <= self.warmup:
                lr = base_lr * e / self.warmup
            else:
                t  = (e - self.warmup) / (self.total - self.warmup)
                lr = self.min_lr + 0.5*(base_lr-self.min_lr)*(1+math.cos(math.pi*t))
            grp["lr"] = lr
    def get_last_lr(self):
        return [g["lr"] for g in self.opt.param_groups]

print("Loss, MixUp & utilities defined ✓")

Loss, MixUp & utilities defined ✓


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, scheduler, device):
    model.train()
    total_loss, n_correct, n_total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for step, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # MixUp
        mixed_x, y_a, y_b, lam = mixup_data(imgs, labels)

        with autocast(enabled=CFG.MIXED_PREC):
            logits = model(mixed_x)
            loss   = mixup_criterion(criterion, logits, y_a, y_b, lam)
            loss   = loss / CFG.ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step+1) % CFG.ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad()

        total_loss += loss.item() * CFG.ACCUM_STEPS
        with torch.no_grad():
            preds = logits.argmax(1)
        n_correct += (preds == labels).sum().item()
        n_total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    scheduler.step()
    return (total_loss/len(loader),
            n_correct/n_total,
            f1_score(all_labels, all_preds, average="macro"))

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n_correct, n_total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=CFG.MIXED_PREC):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        preds      = logits.argmax(1)
        n_correct += (preds == labels).sum().item()
        n_total   += labels.size(0)
        total_loss += loss.item()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss/len(loader), n_correct/n_total, macro_f1, all_preds, all_labels

print("Training loop defined ✓")

Training loop defined ✓


In [ ]:
def run_training(train_df: pd.DataFrame, model_cfg: dict):
    model_name = model_cfg["name"]
    img_size   = model_cfg["img_size"]
    safe_name  = model_name.replace("/","_").replace(".","_")
    oof_preds  = np.zeros((len(train_df), CFG.NUM_CLASSES), dtype=np.float32)

    print(f"\n{'='*62}")
    print(f" Training: {model_name}  |  img_size={img_size}")
    print(f"{'='*62}")

    for fold in CFG.TRAIN_FOLDS:
        print(f"\n── Fold {fold+1}/{CFG.N_FOLDS} ──────────────────────────────")
        tr_df = train_df[train_df.fold != fold].reset_index(drop=True)
        vl_df = train_df[train_df.fold == fold].reset_index(drop=True)

        tr_ds = PigDataset(tr_df, CFG.TRAIN_IMG, get_train_transforms(img_size))
        vl_ds = PigDataset(vl_df, CFG.TRAIN_IMG, get_val_transforms(img_size))

        sampler   = make_weighted_sampler(tr_df["class_id"].tolist())
        tr_loader = DataLoader(tr_ds, batch_size=CFG.BATCH_SIZE, sampler=sampler,
                               num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY,
                               drop_last=True)
        vl_loader = DataLoader(vl_ds, batch_size=CFG.BATCH_SIZE*2, shuffle=False,
                               num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)

        model     = PigClassifier(model_name, CFG.NUM_CLASSES).to(DEVICE)
        criterion = LabelSmoothingCE(CFG.NUM_CLASSES, CFG.LABEL_SMOOTH)
        optimizer = AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
        scheduler = WarmupCosineScheduler(optimizer, CFG.WARMUP_EPOCHS, CFG.EPOCHS)
        scaler    = GradScaler(enabled=CFG.MIXED_PREC)

        best_f1, best_state = 0.0, None
        soup_states = []   # for model soup

        for epoch in range(1, CFG.EPOCHS+1):
            tr_loss, tr_acc, tr_f1 = train_one_epoch(
                model, tr_loader, optimizer, criterion, scaler, scheduler, DEVICE)
            vl_loss, vl_acc, vl_f1, _, _ = validate(
                model, vl_loader, criterion, DEVICE)

            lr_now = scheduler.get_last_lr()[0]
            marker = "  ✓" if vl_f1 > best_f1 else ""
            print(f"  Ep{epoch:02d}/{CFG.EPOCHS} lr={lr_now:.1e}"
                  f"  tr_f1={tr_f1:.4f}  vl_f1={vl_f1:.4f}{marker}")

            if vl_f1 > best_f1:
                best_f1    = vl_f1
                best_state = deepcopy(model.state_dict())

            # collect last N epochs for model soup
            if epoch >= CFG.EPOCHS - CFG.SOUP_EPOCHS + 1:
                soup_states.append(deepcopy(model.state_dict()))

        # ── Model Soup ───────────────────────────────────────────────────────
        if len(soup_states) > 1:
            soup_state = {}
            for key in soup_states[0].keys():
                soup_state[key] = torch.stack(
                    [s[key].float() for s in soup_states]).mean(0)
            model.load_state_dict(soup_state)
            _, _, soup_f1, _, _ = validate(model, vl_loader, criterion, DEVICE)
            print(f"  Soup F1={soup_f1:.4f}  Best F1={best_f1:.4f}", end="")
            if soup_f1 > best_f1:
                best_state = soup_state
                best_f1    = soup_f1
                print("  → using soup ✓")
            else:
                print("  → using best ckpt ✓")

        print(f"  Final fold F1: {best_f1:.4f}")

        # ── OOF probabilities ─────────────────────────────────────────────────
        model.load_state_dict(best_state)
        model.eval()
        vl_loader2 = DataLoader(vl_ds, batch_size=CFG.BATCH_SIZE*2,
                                shuffle=False, num_workers=CFG.NUM_WORKERS)
        oof_probs = []
        with torch.no_grad():
            for imgs, _ in vl_loader2:
                with autocast(enabled=CFG.MIXED_PREC):
                    probs = F.softmax(model(imgs.to(DEVICE)), dim=1).cpu().numpy()
                oof_probs.append(probs)
        oof_probs = np.concatenate(oof_probs)
        fold_idx  = train_df[train_df.fold == fold].index.tolist()
        oof_preds[fold_idx] = oof_probs

        # ── Save checkpoint ───────────────────────────────────────────────────
        ckpt = os.path.join(CFG.OUTPUT, f"{safe_name}_fold{fold}.pth")
        torch.save(best_state, ckpt)
        print(f"  Saved → {ckpt}")

        del model, optimizer, scheduler, scaler
        del tr_ds, vl_ds, tr_loader, vl_loader, vl_loader2, soup_states
        gc.collect(); torch.cuda.empty_cache()

    # ── OOF summary ───────────────────────────────────────────────────────────
    oof_cls = oof_preds.argmax(1)
    oof_f1  = f1_score(train_df["class_id"], oof_cls, average="macro")
    print(f"\n OOF Macro F1 [{model_name}]: {oof_f1:.4f}")
    print(classification_report(train_df["class_id"], oof_cls,
                                 target_names=CFG.CLASS_NAMES, digits=4))
    np.save(os.path.join(CFG.OUTPUT, f"oof_{safe_name}.npy"), oof_preds)
    return oof_preds, oof_f1

print("K-Fold training defined ✓")

K-Fold training defined ✓


In [ ]:
@torch.no_grad()
def tta_predict_batch(model, crops, img_size, n_tta=CFG.TTA_STEPS):
    model.eval()
    val_tf = get_val_transforms(img_size)
    tta_tf = get_tta_transforms(img_size)
    all_probs = []
    # Clean pass
    t = torch.stack([val_tf(image=c)["image"] for c in crops]).to(DEVICE)
    with autocast(enabled=CFG.MIXED_PREC):
        all_probs.append(F.softmax(model(t), dim=1).cpu().numpy())
    # Stochastic passes
    for _ in range(n_tta - 1):
        t = torch.stack([tta_tf(image=c)["image"] for c in crops]).to(DEVICE)
        with autocast(enabled=CFG.MIXED_PREC):
            all_probs.append(F.softmax(model(t), dim=1).cpu().numpy())
    return np.mean(all_probs, axis=0)

def run_inference(test_df, model_cfg):
    model_name = model_cfg["name"]
    img_size   = model_cfg["img_size"]
    safe_name  = model_name.replace("/","_").replace(".","_")
    INFER_BS   = 128
    print(f"\nInference: {model_name}")
    fold_probs = []

    for fold in CFG.TRAIN_FOLDS:
        ckpt = os.path.join(CFG.OUTPUT, f"{safe_name}_fold{fold}.pth")
        if not os.path.exists(ckpt):
            print(f"  WARN: {ckpt} not found"); continue
        model = PigClassifier(model_name, CFG.NUM_CLASSES, pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model.eval()

        all_probs = []
        for start in range(0, len(test_df), INFER_BS):
            batch = test_df.iloc[start:start+INFER_BS]
            crops = []
            for _, row in batch.iterrows():
                img = cv2.imread(os.path.join(CFG.TEST_IMG, row.image_id))
                img = np.zeros((256,256,3),dtype=np.uint8) if img is None \
                      else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                c = crop_with_padding(img, row.xmin, row.ymin, row.w, row.h)
                if c.shape[0]<16 or c.shape[1]<16:
                    c = cv2.resize(c,(64,64))
                crops.append(c)
            all_probs.append(tta_predict_batch(model, crops, img_size))

        fold_probs.append(np.concatenate(all_probs))
        print(f"  fold {fold} ✓")
        del model; gc.collect(); torch.cuda.empty_cache()

    avg = np.mean(fold_probs, axis=0)
    np.save(os.path.join(CFG.OUTPUT, f"test_probs_{safe_name}.npy"), avg)
    return avg

print("Inference defined ✓")

Inference defined ✓


In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TIMM_FUSED_ATTN"] = "0"

# Pre-download with timeout retry
import timm.models._hub as timm_hub
import huggingface_hub
huggingface_hub.constants.HF_HUB_DOWNLOAD_TIMEOUT = 120

In [ ]:
all_oof_preds  = []
all_test_probs = []
weights        = []

for model_cfg in CFG.MODELS:
    oof_preds, oof_f1 = run_training(train_df, model_cfg)
    test_probs        = run_inference(test_df, model_cfg)
    all_oof_preds.append(oof_preds)
    all_test_probs.append(test_probs)
    weights.append(model_cfg["weight"])

print("\n✅ Session 1 training complete")


 Training: tf_efficientnetv2_m  |  img_size=320

── Fold 1/5 ──────────────────────────────


model.safetensors:   0%|          | 0.00/218M [00:00<?, ?B/s]

  Ep01/30 lr=6.7e-05  tr_f1=0.4003  vl_f1=0.7638  ✓
  Ep02/30 lr=1.3e-04  tr_f1=0.5140  vl_f1=0.8315  ✓
  Ep03/30 lr=2.0e-04  tr_f1=0.5398  vl_f1=0.8509  ✓
  Ep04/30 lr=2.0e-04  tr_f1=0.5491  vl_f1=0.8294
  Ep05/30 lr=2.0e-04  tr_f1=0.5726  vl_f1=0.8545  ✓
  Ep06/30 lr=1.9e-04  tr_f1=0.5601  vl_f1=0.8576  ✓
  Ep07/30 lr=1.9e-04  tr_f1=0.5602  vl_f1=0.8654  ✓
  Ep08/30 lr=1.8e-04  tr_f1=0.5908  vl_f1=0.8863  ✓
  Ep09/30 lr=1.8e-04  tr_f1=0.5849  vl_f1=0.8900  ✓
  Ep10/30 lr=1.7e-04  tr_f1=0.5650  vl_f1=0.8998  ✓
  Ep11/30 lr=1.6e-04  tr_f1=0.6021  vl_f1=0.8850
  Ep12/30 lr=1.5e-04  tr_f1=0.5858  vl_f1=0.8963
  Ep13/30 lr=1.4e-04  tr_f1=0.5966  vl_f1=0.8839
  Ep14/30 lr=1.3e-04  tr_f1=0.6052  vl_f1=0.8979
  Ep15/30 lr=1.2e-04  tr_f1=0.5868  vl_f1=0.9070  ✓
  Ep16/30 lr=1.1e-04  tr_f1=0.5843  vl_f1=0.9103  ✓
  Ep17/30 lr=9.5e-05  tr_f1=0.6151  vl_f1=0.8955
  Ep18/30 lr=8.3e-05  tr_f1=0.5887  vl_f1=0.9117  ✓
  Ep19/30 lr=7.2e-05  tr_f1=0.5848  vl_f1=0.9119  ✓
  Ep20/30 lr=6.1e-05  tr_f1=0.

In [ ]:
from sklearn.metrics import confusion_matrix

# Quick ensemble with default weights
w_total  = sum(weights)
ens_oof  = sum(p*(w/w_total) for p,w in zip(all_oof_preds, weights))
oof_cls  = ens_oof.argmax(1)
oof_f1   = f1_score(train_df["class_id"], oof_cls, average="macro")
print(f"Ensemble OOF F1: {oof_f1:.4f}")

cm = confusion_matrix(train_df["class_id"], oof_cls)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CFG.CLASS_NAMES,
            yticklabels=CFG.CLASS_NAMES)
plt.title(f"OOF Confusion Matrix  F1={oof_f1:.4f}")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

print(classification_report(train_df["class_id"], oof_cls,
                             target_names=CFG.CLASS_NAMES, digits=4))

In [ ]:
best_w, best_f1 = None, 0.0

if len(all_oof_preds) == 3:
    for w1 in np.arange(0.2, 0.6, 0.05):
        for w2 in np.arange(0.2, 0.6, 0.05):
            w3 = round(1.0 - w1 - w2, 4)
            if w3 < 0.1 or w3 > 0.6: continue
            ens = (all_oof_preds[0]*w1 + all_oof_preds[1]*w2 + all_oof_preds[2]*w3)
            f1  = f1_score(train_df["class_id"], ens.argmax(1), average="macro")
            if f1 > best_f1:
                best_f1 = f1
                best_w  = (round(float(w1),3), round(float(w2),3), round(float(w3),3))
    print(f"Best OOF F1 : {best_f1:.4f}")
    print(f"Best weights: {best_w}")
else:
    for w1 in np.arange(0.1, 1.0, 0.05):
        w2  = round(1.0 - w1, 4)
        ens = all_oof_preds[0]*w1 + all_oof_preds[1]*w2
        f1  = f1_score(train_df["class_id"], ens.argmax(1), average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_w  = (round(float(w1),3), round(float(w2),3))
    print(f"Best OOF F1 : {best_f1:.4f}")
    print(f"Best weights: {best_w}")

In [ ]:
def apply_temperature(probs, temperature):
    logits = np.log(probs + 1e-8) / temperature
    exp    = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)

best_temp, best_f1_temp = 1.0, 0.0

for temp in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5, 2.0]:
    if len(best_w) == 3:
        w1,w2,w3 = best_w
        ens = (apply_temperature(all_oof_preds[0], temp)*w1 +
               apply_temperature(all_oof_preds[1], temp)*w2 +
               apply_temperature(all_oof_preds[2], temp)*w3)
    else:
        w1,w2 = best_w
        ens = (apply_temperature(all_oof_preds[0], temp)*w1 +
               apply_temperature(all_oof_preds[1], temp)*w2)
    f1 = f1_score(train_df["class_id"], ens.argmax(1), average="macro")
    print(f"  temp={temp:.1f}  OOF F1={f1:.4f}")
    if f1 > best_f1_temp:
        best_f1_temp = f1
        best_temp    = temp

print(f"\n✅ Best temperature : {best_temp}")
print(f"✅ Best OOF F1      : {best_f1_temp:.4f}")

In [ ]:
def build_ensemble_probs(oof_list, test_list, weights, temperature):
    w_total   = sum(weights)
    ens_oof   = sum(apply_temperature(p, temperature)*(w/w_total)
                    for p,w in zip(oof_list, weights))
    ens_test  = sum(apply_temperature(p, temperature)*(w/w_total)
                    for p,w in zip(test_list, weights))
    return ens_oof, ens_test

current_train      = train_df.copy()
current_oof_list   = [p.copy() for p in all_oof_preds]
current_test_list  = [p.copy() for p in all_test_probs]
current_weights    = list(weights)

for round_num, confidence in enumerate(CFG.PSEUDO_ROUNDS, 1):
    print(f"\n{'='*62}")
    print(f" PSEUDO LABEL ROUND {round_num}  — confidence >= {confidence}")
    print(f"{'='*62}")

    _, ens_test = build_ensemble_probs(
        current_oof_list, current_test_list,
        current_weights, best_temp)

    max_probs      = ens_test.max(axis=1)
    confident_mask = max_probs >= confidence
    pseudo_cls     = ens_test[confident_mask].argmax(1)

    print(f"Pseudo labels added : {confident_mask.sum():,} / {len(test_df):,}")
    for i, name in enumerate(CFG.CLASS_NAMES):
        print(f"  {i} {name}: {(pseudo_cls==i).sum():,}")

    pseudo_df = test_df[confident_mask].copy().reset_index(drop=True)
    pseudo_df["class_id"] = pseudo_cls

    combined_df = pd.concat([train_df, pseudo_df], ignore_index=True)
    combined_df = add_fold_column(combined_df)

    print(f"\nTotal training size: {len(combined_df):,}")

    # Retrain all models on combined data
    new_oof_list  = []
    new_test_list = []

    for model_cfg in CFG.MODELS:
        oof_new, _ = run_training(combined_df, model_cfg)
        tst_new    = run_inference(test_df, model_cfg)

        # OOF only valid for real train rows
        real_oof = oof_new[:len(train_df)]
        new_oof_list.append(real_oof)
        new_test_list.append(tst_new)

    current_train     = combined_df
    current_oof_list  = new_oof_list
    current_test_list = new_test_list

    # Re-optimize weights on new OOF
    best_w_r, best_f1_r = None, 0.0
    if len(current_oof_list) == 3:
        for w1 in np.arange(0.2, 0.6, 0.05):
            for w2 in np.arange(0.2, 0.6, 0.05):
                w3 = round(1.0-w1-w2, 4)
                if w3<0.1 or w3>0.6: continue
                ens = (current_oof_list[0]*w1 + current_oof_list[1]*w2
                       + current_oof_list[2]*w3)
                f1  = f1_score(train_df["class_id"], ens.argmax(1), average="macro")
                if f1 > best_f1_r:
                    best_f1_r = f1
                    best_w_r  = (round(float(w1),3), round(float(w2),3), round(float(w3),3))
    else:
        for w1 in np.arange(0.1,1.0,0.05):
            w2  = round(1.0-w1,4)
            ens = current_oof_list[0]*w1 + current_oof_list[1]*w2
            f1  = f1_score(train_df["class_id"], ens.argmax(1), average="macro")
            if f1 > best_f1_r:
                best_f1_r = f1
                best_w_r  = (round(float(w1),3), round(float(w2),3))

    best_w = best_w_r
    print(f"\nRound {round_num} OOF F1 : {best_f1_r:.4f}")
    print(f"Round {round_num} weights : {best_w}")

print("\n✅ Pseudo labeling complete")

In [ ]:
# Build final ensemble with optimized weights + temperature
if len(best_w) == 3:
    w1,w2,w3 = best_w
    ens_final = (apply_temperature(current_test_list[0], best_temp)*w1 +
                 apply_temperature(current_test_list[1], best_temp)*w2 +
                 apply_temperature(current_test_list[2], best_temp)*w3)
else:
    w1,w2 = best_w
    ens_final = (apply_temperature(current_test_list[0], best_temp)*w1 +
                 apply_temperature(current_test_list[1], best_temp)*w2)

test_preds = ens_final.argmax(1)
sub_df = pd.DataFrame({
    "row_id"   : test_df["row_id"],
    "class_id" : test_preds
})

# Save locally
local_path = os.path.join(CFG.OUTPUT, "submission_final.csv")
sub_df.to_csv(local_path, index=False)

# Save to Google Drive
drive_path = os.path.join(CFG.ROOT, "submission_final.csv")
sub_df.to_csv(drive_path, index=False)

print(f"✅ Saved locally → {local_path}")
print(f"✅ Saved to root → {drive_path}")
print(f"Rows : {len(sub_df):,}")
print("\nPredicted distribution:")
for i, name in enumerate(CFG.CLASS_NAMES):
    cnt = (sub_df.class_id == i).sum()
    pct = cnt / len(sub_df) * 100
    print(f"  {i} {name:22s}: {cnt:5,}  ({pct:.1f}%)")
sub_df.head(10)

In [ ]:
import os
output_path = os.path.join(CFG.OUTPUT, 'submission_final.csv')
if os.path.exists(output_path):
    print(f'✅ Found final output at: {output_path}')
    display(pd.read_csv(output_path).head())
else:
    print(f'❌ File not found at {output_path}. Make sure you have executed the final inference cell (iEM0ZkUyhPQt).')